In [ ]:
%pip install lightgbm catboost scikit-learn pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [1]:
# =====================================================================
# ECOGRID AI: MULTI-TASK HETEROGENEOUS GRADIENT BOOSTING ENSEMBLE
# Production Build | Fully Self-Explaining Console Output Engine
# =====================================================================

import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend to force rendering image files
import matplotlib.pyplot as plt

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

def console_log(msg: str, delay: float = 0.01):
    """Outputs structured log messages directly to standard output."""
    print(msg, flush=True)
    time.sleep(delay)

class EcoGridEngine:
    def __init__(self):
        self.label_encoder = LabelEncoder()
        self.feature_cols = ['Hour_Sin', 'Hour_Cos', 'DayOfWeek', 'IsWeekend', 'Ambient_Temp_C', 'Temp_Rolling_Mean']

        # Hyperparameters with L2 Regularization & Tree Depth Constraints
        self.lgb_params = {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.03, 'reg_lambda': 8.0, 'random_state': 42, 'verbose': -1}
        self.cat_params = {'iterations': 100, 'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 8.0, 'random_state': 42, 'verbose': 0}

    def run_pipeline(self):
        console_log("="*80)
        console_log(" 🚀 ECOGRID AI: MULTI-TASK GRADIENT BOOSTING PIPELINE")
        console_log("="*80)

        # 1. Telemetry Simulation & Cyclical Feature Engineering
        console_log("\n[STEP 1/4] Executing Data Engineering & Cyclical Transformations...")
        date_range = pd.date_range(start="2026-07-01", periods=720, freq="h")
        df = pd.DataFrame({"Timestamp": date_range})
        df["Hour"] = df["Timestamp"].dt.hour
        df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
        df["IsWeekend"] = df["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)
        df["Ambient_Temp_C"] = 28 + 6 * np.sin(2 * np.pi * df["Hour"] / 24) + np.random.normal(0, 1.2, len(df))

        # Sine/Cosine Transformation for Time Boundaries
        df["Hour_Sin"] = np.sin(2 * np.pi * df["Hour"] / 24.0)
        df["Hour_Cos"] = np.cos(2 * np.pi * df["Hour"] / 24.0)
        df["Temp_Rolling_Mean"] = df["Ambient_Temp_C"].rolling(window=3, min_periods=1).mean()

        def assign_occupancy(row):
            if row["IsWeekend"] == 1: return "Low"
            return np.random.choice(["High", "Medium", "Low"], p=[0.5, 0.3, 0.2]) if 9 <= row["Hour"] <= 17 else "Low"

        df["Occupancy_State"] = df.apply(assign_occupancy, axis=1)
        df["Occupancy_Label"] = self.label_encoder.fit_transform(df["Occupancy_State"])

        occupancy_weights = {"High": 28.5, "Medium": 14.0, "Low": 3.5}
        df["HVAC_Power_kW"] = df.apply(
            lambda r: round(12.0 + occupancy_weights[r["Occupancy_State"]] + max(0, (r["Ambient_Temp_C"] - 26) * 2.1) + np.random.normal(0, 1.0), 2),
            axis=1
        )

        # 2. Strict Sequential Partitioning (No Time-Series Leakage)
        X = df[self.feature_cols]
        y_cls = df["Occupancy_Label"]
        y_reg = df["HVAC_Power_kW"]

        X_train, X_test, y_train_cls, y_test_cls = train_test_split(X, y_cls, test_size=0.2, shuffle=False)
        _, _, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, shuffle=False)

        console_log(f" ↳ Dataset: {len(df)} Rows | Train Window: {len(X_train)} Hours | Evaluation Horizon: {len(X_test)} Hours")

        # 3. Model Training
        console_log("\n[STEP 2/4] Fitting Dual-Track Heterogeneous Ensemble...")
        m1_cls = lgb.LGBMClassifier(**self.lgb_params).fit(X_train, y_train_cls)
        m2_cls = CatBoostClassifier(**self.cat_params).fit(X_train, y_train_cls)

        m1_reg = lgb.LGBMRegressor(**self.lgb_params).fit(X_train, y_train_reg)
        m2_reg = CatBoostRegressor(**self.cat_params).fit(X_train, y_train_reg)

        # 4. Metric Extraction
        console_log("\n[STEP 3/4] Evaluating Model Generalization Metrics...")
        tr_prob = (m1_cls.predict_proba(X_train) + m2_cls.predict_proba(X_train)) / 2
        te_prob = (m1_cls.predict_proba(X_test) + m2_cls.predict_proba(X_test)) / 2
        train_acc = accuracy_score(y_train_cls, np.argmax(tr_prob, axis=1))
        test_acc = accuracy_score(y_test_cls, np.argmax(te_prob, axis=1))

        te_pred_reg = (m1_reg.predict(X_test) + m2_reg.predict(X_test)) / 2
        tr_pred_reg = (m1_reg.predict(X_train) + m2_reg.predict(X_train)) / 2
        train_rmse = np.sqrt(mean_squared_error(y_train_reg, tr_pred_reg))
        test_rmse = np.sqrt(mean_squared_error(y_test_reg, te_pred_reg))

        console_log(f" 📑 Classification Accuracy -> Train: {train_acc*100:.1f}% | Test: {test_acc*100:.1f}%")
        console_log(f" 📑 Forecasting RMSE       -> Train: {train_rmse:.2f} kW | Test: {test_rmse:.2f} kW")

        # Live Endpoint Test
        self.live_api_endpoint(m1_cls, m2_cls, m1_reg, m2_reg, hour=14, day_of_week=2, temp=34.5, temp_avg=33.8)

        # Generate Chart Output File
        self.plot_and_save(df, y_test_reg, te_pred_reg)

    def live_api_endpoint(self, m1_cls, m2_cls, m1_reg, m2_reg, hour, day_of_week, temp, temp_avg):
        console_log("\n[STEP 4/4] Executing Live Production API Emulation...")
        hour_sin = np.sin(2 * np.pi * hour / 24.0)
        hour_cos = np.cos(2 * np.pi * hour / 24.0)
        is_weekend = 1 if day_of_week >= 5 else 0

        payload = pd.DataFrame([[hour_sin, hour_cos, day_of_week, is_weekend, temp, temp_avg]], columns=self.feature_cols)

        voted_probs = (m1_cls.predict_proba(payload) + m2_cls.predict_proba(payload)) / 2
        assigned_state = self.label_encoder.classes_[np.argmax(voted_probs, axis=1)[0]]

        blended_pred = (m1_reg.predict(payload)[0] + m2_reg.predict(payload)[0]) / 2

        console_log(f" ↳ Incoming Telemetry --> 14:00 | Wednesday | Temp: {temp}°C")
        console_log(f" ↳ Spatial Model State --> [{assigned_state.upper()}] Occupancy")
        console_log(f" ↳ Kinetic Forecast    --> [{blended_pred:.2f} kW] Electrical Load")

        if assigned_state == "High" and blended_pred > 40.0:
            console_log(" 🚨 ROUTING ACTION    --> PRE-COOLING ENGAGED. Buffering peak spike.")
        elif assigned_state == "Low":
            console_log(" 🍃 ROUTING ACTION    --> DEEP HIBERNATE MODE ENGAGED.")
        else:
            console_log(" ⚖️ ROUTING ACTION    --> STEADY STATE MAINTAINED.")
        console_log("="*80)

    def plot_and_save(self, df, y_test_reg, te_pred_reg):
        plt.figure(figsize=(12, 5))
        plt.plot(df["Timestamp"].iloc[-120:].values, y_test_reg[:120].values, label="Actual Load (kW)", color="cyan", alpha=0.8)
        plt.plot(df["Timestamp"].iloc[-120:].values, te_pred_reg[:120], label="EcoGrid Predicted Load (kW)", color="magenta", linestyle="--")
        plt.title("EcoGrid Intelligent Optimization Engine: 5-Day Horizon Load Prediction")
        plt.xlabel("Timeline")
        plt.ylabel("Infrastructural Load (kW)")
        plt.legend()
        plt.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.savefig("ecogrid_horizon.png")
        console_log(" 🖼️ Output Graph Saved Successfully: 'ecogrid_horizon.png'")

if __name__ == "__main__":
    engine = EcoGridEngine()
    engine.run_pipeline()

 🚀 ECOGRID AI: MULTI-TASK GRADIENT BOOSTING PIPELINE

[STEP 1/4] Executing Data Engineering & Cyclical Transformations...
 ↳ Dataset: 720 Rows | Train Window: 576 Hours | Evaluation Horizon: 144 Hours

[STEP 2/4] Fitting Dual-Track Heterogeneous Ensemble...

[STEP 3/4] Evaluating Model Generalization Metrics...
 📑 Classification Accuracy -> Train: 85.6% | Test: 84.7%
 📑 Forecasting RMSE       -> Train: 5.38 kW | Test: 5.81 kW

[STEP 4/4] Executing Live Production API Emulation...
 ↳ Incoming Telemetry --> 14:00 | Wednesday | Temp: 34.5°C
 ↳ Spatial Model State --> [LOW] Occupancy
 ↳ Kinetic Forecast    --> [39.42 kW] Electrical Load
 🍃 ROUTING ACTION    --> DEEP HIBERNATE MODE ENGAGED.
 🖼️ Output Graph Saved Successfully: 'ecogrid_horizon.png'
